# 03 – Modeling

Train a **Logistic Regression** and an **XGBoost** classifier on the preprocessed Telco churn data.

Inputs (from `02_preprocessing.ipynb`):
- `data/processed/train.csv` – feature matrix + `Churn` label (train split)
- `data/processed/test.csv`  – feature matrix + `Churn` label (test split)

Outputs:
- `models/logistic_regression.joblib`
- `models/xgboost.joblib`

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

ROOT = Path("..")
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

## 1. Load data

In [2]:
train = pd.read_csv(ROOT / "data/processed/train.csv")
test  = pd.read_csv(ROOT / "data/processed/test.csv")

X_train = train.drop(columns=["Churn"])
y_train = train["Churn"]
X_test  = test.drop(columns=["Churn"])
y_test  = test["Churn"]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Churn rate – train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

Train: (5634, 45)  |  Test: (1409, 45)
Churn rate – train: 0.265  test: 0.265


## 2. Train models

In [3]:
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [4]:
neg, pos = np.bincount(y_train)
xgb = XGBClassifier(
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
)
xgb.fit(X_train, y_train)

XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=2.768227848101266)

## 3. Evaluation metrics

In [5]:
def evaluate(name, model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y, y_pred), 4),
        "Precision": round(precision_score(y, y_pred), 4),
        "Recall":    round(recall_score(y, y_pred), 4),
        "F1":        round(f1_score(y, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y, y_prob), 4),
    }

metrics = pd.DataFrame([
    evaluate("Logistic Regression", lr,  X_test, y_test),
    evaluate("XGBoost",             xgb, X_test, y_test),
]).set_index("Model")

metrics

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Logistic Regression,0.7381,0.5043,0.7834,0.6136,0.8413
XGBoost,0.7523,0.5263,0.6684,0.5889,0.8187


## 4. Confusion matrices

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, model, name in zip(
    axes,
    [lr, xgb],
    ["Logistic Regression", "XGBoost"],
):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=["No Churn", "Churn"],
        cmap="Blues",
        ax=ax,
    )
    ax.set_title(name)

plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save model artifacts

In [7]:
joblib.dump(lr,  MODELS_DIR / "logistic_regression.joblib")
joblib.dump(xgb, MODELS_DIR / "xgboost.joblib")
print("Models saved to", MODELS_DIR)

Models saved to ..\models
